In [2]:
import os
import json
import itertools
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report

import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau, CSVLogger


# Reproduzierbarkeit
SEED = 42
tf.keras.utils.set_random_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

In [3]:
# ── Konfiguration ──────────────────────────────────────────────────────────────
# Passe diesen Pfad bei Bedarf an.
DATA_DIR = Path(r"c:/Users/benke/Downloads/archive/Fruit And Vegetable Diseases Dataset/Apple_Data_sing")

IMG_SIZE   = (224, 224)
BATCH_SIZE = 32
SEED       = 42



In [4]:
if not DATA_DIR.exists():
    raise FileNotFoundError(
        f"DATA_DIR existiert nicht: {DATA_DIR}\n"
        "Passe den Pfad in der Konfigurationszelle an."
    )

train_ds_raw = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.3,
    subset="training",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

temp_ds_raw = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.3,
    subset="validation",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

class_names = train_ds_raw.class_names
num_classes = len(class_names)

# Validation-Teil wird halbiert: 15% Validation, 15% Test
# Wichtig: Testdaten werden erst ganz am Ende zur Bewertung verwendet.
temp_batches = temp_ds_raw.cardinality().numpy()
val_batches = temp_batches // 2

val_ds_raw = temp_ds_raw.take(val_batches)
test_ds_raw = temp_ds_raw.skip(val_batches)

print("Klassen:", class_names)
print("Anzahl Klassen:", num_classes)
print("Training Batches:", train_ds_raw.cardinality().numpy())
print("Validation Batches:", val_ds_raw.cardinality().numpy())
print("Test Batches:", test_ds_raw.cardinality().numpy())
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.2),
    layers.RandomZoom(0.1),
    layers.RandomBrightness(0.2),
    layers.RandomContrast(0.1),
], name="augmentation")

AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds_raw.map(
    lambda x, y: (data_augmentation(x, training=True), y),
    num_parallel_calls=AUTOTUNE
).prefetch(AUTOTUNE)

val_ds = val_ds_raw.prefetch(AUTOTUNE)
test_ds = test_ds_raw.prefetch(AUTOTUNE)

Found 4817 files belonging to 2 classes.
Using 3372 files for training.
Found 4817 files belonging to 2 classes.
Using 1445 files for validation.
Klassen: ['Apple__Healthy', 'Apple__Rotten']
Anzahl Klassen: 2
Training Batches: 106
Validation Batches: 23
Test Batches: 23


In [5]:
from tensorflow.keras.preprocessing import image as keras_image
import xgboost as xgb
import joblib

#LADEN eigenes mini cnn
model_simpel = tf.keras.models.load_model("simplemodel.keras")

#LADEN randomforest und xg boost (+laden mobilenetv2, +extrahierte features)
mobilenet_extr = tf.keras.models.load_model("mobilenet_extractor.keras")
xgb_model = xgb.XGBClassifier()
xgb_model.load_model("mobilenet_xgboost.json") 
model_randomforest = joblib.load("rf_classifier.joblib")

X_train = np.load("X_train.npy")
y_train = np.load("y_train.npy")
X_val = np.load("X_val.npy")
y_val = np.load("y_val.npy")

#LADEN optimiertes pretrained CNN
model_best_CNN =  tf.keras.models.load_model("candidate_03.keras")

#LADEN optimiertes finetuned pretrained CNN
model_finetuned = tf.keras.models.load_model("best_correct_finetuned_model.keras")

In [54]:
import time
model = model_simpel
print("👾model_simpel👾")
print("=== Time ===")
images_batch, _ = next(iter(test_ds))
# Warm-up: Feature-Extraktion + Random-Forest-Vorhersage
for _ in range(5):
    model.predict(images_batch, verbose=0)


n_runs = 100
start = time.time()

for _ in range(n_runs):
    model.predict(images_batch, verbose=0)

end = time.time()

# Zeit pro Bild, nicht nur pro Batch:
avg_time_per_image = (end - start) / (n_runs * len(images_batch))

print(f"Durchschnittliche Laufzeit pro Bild: {avg_time_per_image * 1000:.2f} ms")
y_true, y_pred, y_conf = [], [], []

def get_file_size_mb(path):
    return os.path.getsize(path) / (1024**2)
print("=== Size ===")
size_mb = get_file_size_mb("simplemodel.keras")
print(f"{size_mb:.2f} MB")

for images, labels in test_ds:
    preds = model.predict(images, verbose=0)
    y_true.extend(labels.numpy())
    y_pred.extend(np.argmax(preds, axis=1))
    y_conf.extend(np.max(preds, axis=1))

y_true = np.array(y_true)
y_pred = np.array(y_pred)
y_conf = np.array(y_conf)

print("=== Classification Report ===")
print(classification_report(y_true, y_pred, target_names=class_names))
cm = confusion_matrix(y_true, y_pred)

print(cm)

👾model_simpel👾
=== Time ===
Durchschnittliche Laufzeit pro Bild: 3.94 ms
=== Size ===
0.89 MB
=== Classification Report ===
                precision    recall  f1-score   support

Apple__Healthy       0.86      0.85      0.86       299
 Apple__Rotten       0.89      0.90      0.90       410

      accuracy                           0.88       709
     macro avg       0.88      0.88      0.88       709
  weighted avg       0.88      0.88      0.88       709

[[254  45]
 [ 40 370]]


In [50]:
import time
model = model_best_CNN
print("👾model_best_CNN👾")
print("=== Time ===")
images_batch, _ = next(iter(test_ds))
# Warm-up: Feature-Extraktion + Random-Forest-Vorhersage
for _ in range(5):
    model.predict(images_batch, verbose=0)


n_runs = 100
start = time.time()

for _ in range(n_runs):
    model.predict(images_batch, verbose=0)

end = time.time()

# Zeit pro Bild, nicht nur pro Batch:
avg_time_per_image = (end - start) / (n_runs * len(images_batch))

print(f"Durchschnittliche Laufzeit pro Bild: {avg_time_per_image * 1000:.2f} ms")
y_true, y_pred, y_conf = [], [], []

def get_file_size_mb(path):
    return os.path.getsize(path) / (1024**2)
print("=== Size ===")
size_mb = get_file_size_mb("finetune_03.keras")
print(f"{size_mb:.2f} MB")

for images, labels in test_ds:
    preds = model.predict(images, verbose=0)
    y_true.extend(labels.numpy())
    y_pred.extend(np.argmax(preds, axis=1))
    y_conf.extend(np.max(preds, axis=1))

y_true = np.array(y_true)
y_pred = np.array(y_pred)
y_conf = np.array(y_conf)

print("=== Classification Report ===")
print(classification_report(y_true, y_pred, target_names=class_names))
cm = confusion_matrix(y_true, y_pred)

print(cm)

👾model_best_CNN👾
=== Time ===
Durchschnittliche Laufzeit pro Bild: 19.74 ms
=== Size ===
23.53 MB
=== Classification Report ===
                precision    recall  f1-score   support

Apple__Healthy       0.99      0.99      0.99       286
 Apple__Rotten       0.99      0.99      0.99       423

      accuracy                           0.99       709
     macro avg       0.99      0.99      0.99       709
  weighted avg       0.99      0.99      0.99       709

[[282   4]
 [  4 419]]


In [51]:
import time
model = model_finetuned
print("👾model_finetuned👾")
print("=== Time ===")
images_batch, _ = next(iter(test_ds))
# Warm-up: Feature-Extraktion + Random-Forest-Vorhersage
for _ in range(5):
    model.predict(images_batch, verbose=0)


n_runs = 100
start = time.time()

for _ in range(n_runs):
    model.predict(images_batch, verbose=0)

end = time.time()

# Zeit pro Bild, nicht nur pro Batch:
avg_time_per_image = (end - start) / (n_runs * len(images_batch))

print(f"Durchschnittliche Laufzeit pro Bild: {avg_time_per_image * 1000:.2f} ms")
y_true, y_pred, y_conf = [], [], []

def get_file_size_mb(path):
    return os.path.getsize(path) / (1024**2)
print("=== Size ===")
size_mb = get_file_size_mb("best_correct_finetuned_model.keras")
print(f"{size_mb:.2f} MB")

for images, labels in test_ds:
    preds = model.predict(images, verbose=0)
    y_true.extend(labels.numpy())
    y_pred.extend(np.argmax(preds, axis=1))
    y_conf.extend(np.max(preds, axis=1))

y_true = np.array(y_true)
y_pred = np.array(y_pred)
y_conf = np.array(y_conf)

print("=== Classification Report ===")
print(classification_report(y_true, y_pred, target_names=class_names))
cm = confusion_matrix(y_true, y_pred)

print(cm)


👾model_finetuned👾
=== Time ===
Durchschnittliche Laufzeit pro Bild: 18.58 ms
=== Size ===
18.98 MB
=== Classification Report ===
                precision    recall  f1-score   support

Apple__Healthy       1.00      0.99      0.99       281
 Apple__Rotten       0.99      1.00      1.00       428

      accuracy                           1.00       709
     macro avg       1.00      0.99      1.00       709
  weighted avg       1.00      1.00      1.00       709

[[278   3]
 [  0 428]]


In [11]:
def get_file_size_mb(path):
    return os.path.getsize(path) / (1024 ** 2)


print("\n========================================")
print("👾 Modell: Random Forest + MobileNetV2 👾")

# Ein einzelnes Testbild bzw. ein kleiner Batch für die Zeitmessung
images_batch, labels_batch = next(iter(test_ds))

# Wichtig: Features aus dem MobileNet-Extractor erzeugen
features_batch = mobilenet_extr.predict(images_batch, verbose=0)

print("\n=== Time ===")

# Warm-up: Feature-Extraktion + Random-Forest-Vorhersage
for _ in range(5):
    features = mobilenet_extr.predict(images_batch, verbose=0)
    model_randomforest.predict(features)

n_runs = 100
start = time.time()

for _ in range(n_runs):
    features = mobilenet_extr.predict(images_batch, verbose=0)
    prediction = model_randomforest.predict(features)

end = time.time()

# Zeit pro Bild, nicht nur pro Batch:
avg_time_per_image = (end - start) / (n_runs * len(images_batch))

print(f"Durchschnittliche Laufzeit pro Bild: {avg_time_per_image * 1000:.2f} ms")
print("\n=== Size ===")

extractor_size_mb = get_file_size_mb("mobilenet_extractor.keras")
rf_size_mb = get_file_size_mb("rf_classifier.joblib")

total_size_mb = extractor_size_mb + rf_size_mb

print(f"MobileNetV2 Feature Extractor: {extractor_size_mb:.2f} MB")
print(f"Random Forest:                {rf_size_mb:.2f} MB")
print(f"Gesamtmodell:                 {total_size_mb:.2f} MB")
print("\n=== Classification Report ===")

y_true = []
y_pred = []
y_conf = []

for images, labels in test_ds:

    # 1. Bilder in Feature-Vektoren umwandeln
    features = mobilenet_extr.predict(images, verbose=0)

    # 2. Random Forest klassifizieren lassen
    preds = model_randomforest.predict(features)

    # 3. Wahrscheinlichkeiten für die Konfidenz
    probs = model_randomforest.predict_proba(features)

    y_true.extend(labels.numpy())
    y_pred.extend(preds)
    y_conf.extend(np.max(probs, axis=1))

y_true = np.array(y_true)
y_pred = np.array(y_pred)
y_conf = np.array(y_conf)

print(classification_report(
    y_true,
    y_pred,
    target_names=class_names
))
cm = confusion_matrix(y_true, y_pred)

print(cm)



👾 Modell: Random Forest + MobileNetV2 👾

=== Time ===
Durchschnittliche Laufzeit pro Bild: 22.03 ms

=== Size ===
MobileNetV2 Feature Extractor: 9.17 MB
Random Forest:                8.60 MB
Gesamtmodell:                 17.77 MB

=== Classification Report ===
                precision    recall  f1-score   support

Apple__Healthy       0.95      0.98      0.96       295
 Apple__Rotten       0.99      0.96      0.97       414

      accuracy                           0.97       709
     macro avg       0.97      0.97      0.97       709
  weighted avg       0.97      0.97      0.97       709

[[289   6]
 [ 16 398]]


In [10]:
import time
def get_file_size_mb(path):
    return os.path.getsize(path) / (1024 ** 2)
print("\n========================================")
print("👾 Modell: XGBoost + MobileNetV2 Features 👾")

images_batch, _ = next(iter(test_ds))
images_batch = images_batch[0:1]

print("\n=== Time (realistisch, mit model()) ===")

# Warm-up — WICHTIG bei model()-Aufruf, da TF beim ersten Call
# den Graphen tracen/kompilieren muss
for _ in range(10):
    features = mobilenet_extr(images_batch, training=False).numpy()
    _ = xgb_model.predict(features)

n_runs = 100
mobilenet_times = []
xgb_times = []

for _ in range(n_runs):
    t0 = time.perf_counter()
    features = mobilenet_extr(images_batch, training=False).numpy()
    t1 = time.perf_counter()
    prediction = xgb_model.predict(features)
    t2 = time.perf_counter()

    mobilenet_times.append(t1 - t0)
    xgb_times.append(t2 - t1)

avg_mobilenet = np.mean(mobilenet_times) * 1000
avg_xgb = np.mean(xgb_times) * 1000
avg_total = avg_mobilenet + avg_xgb

print(f"MobileNet Feature-Extraktion: {avg_mobilenet:.2f} ms (± {np.std(mobilenet_times)*1000:.2f} ms)")
print(f"XGBoost Prediction:            {avg_xgb:.2f} ms (± {np.std(xgb_times)*1000:.2f} ms)")
print(f"Gesamt pro Bild:                {avg_total:.2f} ms")
print("\n=== Size ===")

extractor_size_mb = get_file_size_mb("mobilenet_extractor.keras")
xgb_size_mb = get_file_size_mb("mobilenet_xgboost.json")

print(f"MobileNetV2 Feature Extractor: {extractor_size_mb:.2f} MB")
print(f"XGBoost-Modell:               {xgb_size_mb:.2f} MB")
print(f"Gesamtmodell:                  {extractor_size_mb + xgb_size_mb:.2f} MB")


print("\n=== Classification Report ===")

y_true = []
y_pred = []
y_conf = []

for images, labels in test_ds:

    # Bilder -> MobileNet-Features
    features = mobilenet_extr.predict(images, verbose=0)

    # XGBoost-Klassifikation
    preds = xgb_model.predict(features)

    # Klassifikationswahrscheinlichkeiten
    probs = xgb_model.predict_proba(features)

    y_true.extend(labels.numpy())
    y_pred.extend(preds.astype(int))
    y_conf.extend(np.max(probs, axis=1))

y_true = np.array(y_true)
y_pred = np.array(y_pred)
y_conf = np.array(y_conf)

print(classification_report(
    y_true,
    y_pred,
    target_names=class_names
))


👾 Modell: XGBoost + MobileNetV2 Features 👾

=== Time (realistisch, mit model()) ===
MobileNet Feature-Extraktion: 204.57 ms (± 13.72 ms)
XGBoost Prediction:            1.15 ms (± 0.43 ms)
Gesamt pro Bild:                205.72 ms

=== Size ===
MobileNetV2 Feature Extractor: 9.17 MB
XGBoost-Modell:               0.58 MB
Gesamtmodell:                  9.75 MB

=== Classification Report ===
                precision    recall  f1-score   support

Apple__Healthy       0.98      0.98      0.98       295
 Apple__Rotten       0.99      0.98      0.99       414

      accuracy                           0.98       709
     macro avg       0.98      0.98      0.98       709
  weighted avg       0.98      0.98      0.98       709



In [15]:
model_finetuned.summary()

Model: "MobileNetV2_Hyperparameteroptimierung"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenet_preprocess            │ (None, 224, 224, 3)    │             0 │
│ (Rescaling)                     │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 1280)           │         5,120 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 2)              │           258 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,826,184 (18.41 MB)

 Trainable params: 1,199,426 (4.58 MB)

 Non-trainable params: 1,227,904 (4.68 MB)

 Optimizer params: 2,398,854 (9.15 MB)

In [17]:
model_best_CNN.summary()

Model: "MobileNetV2_Hyperparameteroptimierung"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenet_preprocess            │ (None, 224, 224, 3)    │             0 │
│ (Rescaling)                     │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 1280)           │         5,120 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 2)              │           258 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,760,904 (10.53 MB)

 Trainable params: 166,786 (651.51 KB)

 Non-trainable params: 2,260,544 (8.62 MB)

 Optimizer params: 333,574 (1.27 MB)

In [ ]:
#speicher = total params - optimiezer params